# Validate the reporting snapshot
This notebook calls the same standard-library quality function as the CLI.
It validates the supplied CSV and does not reproduce industry recovery or ML.
Open from the repository's Git folder in Databricks, or from a local checkout.


In [ ]:
from pathlib import Path
import json
import runpy

# Locate project configuration without an author-specific workspace path.
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in candidates if (p / 'config/portfolio_contract.json').is_file()), None)
if project_root is None:
    raise RuntimeError('Open this notebook from the repository Git folder or checkout.')

# Set a readable file path here if the CSV is in a Unity Catalog Volume.
INPUT_PATH = None
VERIFY_SNAPSHOT = True
input_path = Path(INPUT_PATH) if INPUT_PATH else project_root / 'data/processed/portfolio_reporting.csv'


In [ ]:
module = runpy.run_path(str(project_root / 'src/data_processing/validate_portfolio.py'))
report = module['validate_portfolio'](
    input_path, project_root / 'config/portfolio_contract.json', verify_snapshot=VERIFY_SNAPSHOT
)
print(json.dumps(report, ensure_ascii=False, indent=2))
if report['status'] != 'PASS':
    raise ValueError('Reporting validation failed. Review the issue counts above.')
